In [ ]:
# Docker Data Access Notebook

This notebook provides Docker-compatible data loading for the QTAB factor modeling environment. It's designed to work without requiring individual WRDS credentials, using shared datasets instead.

## Environment Setup

**Important**: This notebook is designed to run in the Docker environment. If you're running locally, adjust the InfluxDB connection parameters accordingly.

### Connection Configuration

```python
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Union
from influxdb_client_3 import InfluxDBClient3, Point, WriteOptions, write_client_options
import os
```

### Docker Environment Detection

```python
# Auto-detect if running in Docker environment
def get_influxdb_host():
    """
    Automatically detect InfluxDB host based on environment
    """
    if os.path.exists('/.dockerenv'):
        # Running inside Docker container
        return "http://influxdb:8086"
    else:
        # Running on host machine
        return "http://localhost:8086"

# Environment configuration
INFLUXDB_HOST = get_influxdb_host()
INFLUXDB_TOKEN = os.getenv('INFLUXDB_TOKEN', 'qtab_factor_token_2024_secure')
INFLUXDB_ORG = os.getenv('INFLUXDB_ORG', 'qtab')
INFLUXDB_BUCKET = os.getenv('INFLUXDB_BUCKET', 'jkp_factors')

print(f"🐳 Environment: {'Docker' if 'influxdb' in INFLUXDB_HOST else 'Local'}")
print(f"📍 InfluxDB Host: {INFLUXDB_HOST}")
print(f"🏢 Organization: {INFLUXDB_ORG}")
print(f"🗄️  Bucket: {INFLUXDB_BUCKET}")
```

## Docker-Compatible Data Loader

```python
class DockerDataLoader:
    """
    Docker-compatible data loader for factor modeling
    Works with shared datasets and pre-processed files
    """
    
    # Factor mappings (same as original but Docker-optimized)
    FACTOR_MODELS = {
        'fama_french_5': {
            'value': ['be_me', 'ni_me', 'ocf_me'],
            'size': ['market_equity'],
            'momentum': ['ret_12_1'],
            'profitability': ['gp_at', 'ni_at'],
            'investment': ['at_gr1', 'oaccruals_at'],
            'market': ['ret_exc', 'beta_60m']
        },
        'quality_comprehensive': {
            'profitability': ['gp_at', 'ni_at', 'ni_be', 'ope_be', 'ocf_at'],
            'quality_composite': ['f_score', 'qmj', 'qmj_prof', 'qmj_safety'],
            'earnings_quality': ['ni_ar1', 'oaccruals_at'],
            'growth_quality': ['sale_gr1', 'at_gr1', 'qmj_growth'],
            'financial_strength': ['at_be', 'cash_at']
        },
        'basic': {
            'returns': ['ret_exc'],
            'size': ['market_equity'],
            'value': ['be_me'],
            'momentum': ['ret_12_1']
        }
    }
    
    def __init__(self, data_directory: str = "/home/jovyan/work/shared-data"):
        """
        Initialize with shared data directory (Docker mount point)
        """
        self.data_dir = data_directory
        self.available_files = self._scan_available_files()
        
    def _scan_available_files(self) -> List[str]:
        """Scan for available data files"""
        import os
        if not os.path.exists(self.data_dir):
            print(f"⚠️  Data directory not found: {self.data_dir}")
            return []
        
        files = [f for f in os.listdir(self.data_dir) 
                if f.endswith(('.csv', '.parquet', '.h5'))]
        print(f"📁 Found {len(files)} data files: {files}")
        return files
    
    def load_shared_characteristics(self, filename: str = "characteristics.csv") -> pd.DataFrame:
        """
        Load characteristics data from shared CSV file
        """
        file_path = os.path.join(self.data_dir, filename)
        
        if not os.path.exists(file_path):
            print(f"❌ File not found: {file_path}")
            print(f"Available files: {self.available_files}")
            return pd.DataFrame()
        
        try:
            # Load the characteristics data
            print(f"📖 Loading data from {filename}...")
            data = pd.read_csv(file_path)
            
            # Basic data validation
            print(f"✅ Loaded {len(data)} observations")
            print(f"📅 Columns: {list(data.columns)}")
            
            if 'eom' in data.columns:
                data['eom'] = pd.to_datetime(data['eom'])
                print(f"📅 Date range: {data['eom'].min()} to {data['eom'].max()}")
            
            return data
            
        except Exception as e:
            print(f"❌ Error loading {filename}: {e}")
            return pd.DataFrame()
    
    def prepare_for_influxdb(self, df: pd.DataFrame, 
                           time_column: str = 'eom',
                           factor_model: str = 'shared_data') -> pd.DataFrame:
        """
        Prepare DataFrame for InfluxDB storage
        """
        if df.empty:
            return df
        
        processed_data = df.copy()
        
        # Set time index
        if time_column in processed_data.columns:
            processed_data = processed_data.set_index(time_column)
        
        # Handle missing values
        categorical_columns = ['size_grp', 'permno', 'gvkey']
        for col in categorical_columns:
            if col in processed_data.columns:
                processed_data[col] = processed_data[col].astype(str).fillna("unknown")
        
        # Ensure numeric columns are properly typed
        numeric_columns = processed_data.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            processed_data[col] = pd.to_numeric(processed_data[col], errors='coerce')
        
        # Add metadata
        processed_data['data_vintage'] = datetime.now().strftime('%Y-%m-%d')
        processed_data['factor_model'] = factor_model
        processed_data['data_source'] = 'shared_csv'
        
        print(f"🔧 Processed data shape: {processed_data.shape}")
        return processed_data
    
    def get_factor_subset(self, df: pd.DataFrame, factor_model: str) -> pd.DataFrame:
        """
        Extract subset of factors for a specific model
        """
        if factor_model not in self.FACTOR_MODELS:
            print(f"❌ Unknown factor model: {factor_model}")
            return df
        
        # Get all characteristics for the model
        characteristics = []
        for factor_group in self.FACTOR_MODELS[factor_model].values():
            characteristics.extend(factor_group)
        
        # Add required columns
        required_cols = ['gvkey', 'permno'] if df.index.name != 'eom' else []
        characteristics.extend(required_cols)
        
        # Filter to available columns
        available_chars = [col for col in characteristics if col in df.columns]
        missing_chars = [col for col in characteristics if col not in df.columns]
        
        if missing_chars:
            print(f"⚠️  Missing characteristics: {missing_chars}")
        
        print(f"📊 Using {len(available_chars)} characteristics for {factor_model}")
        return df[available_chars]

# Initialize Docker data loader
data_loader = DockerDataLoader()
print("🚀 Docker Data Loader initialized")
```

## InfluxDB Connection and Manager

```python
def setup_influxdb_connection():
    """
    Setup InfluxDB connection with proper error handling
    """
    def success_callback(data: str):
        print(f"✅ Successfully wrote batch: {len(data)} records")

    def error_callback(data: str, exception):
        print(f"❌ Failed writing batch: {exception}")

    def retry_callback(data: str, exception):
        print(f"🔄 Retrying write: {exception}")

    # Configure write options for batch processing
    write_options = WriteOptions(
        batch_size=5_000,      # Smaller batches for Docker
        flush_interval=10_000,
        jitter_interval=2_000,
        retry_interval=5_000,
        max_retries=3,
        max_retry_delay=30_000,
        exponential_base=2
    )

    # Create client options
    wco = write_client_options(
        success_callback=success_callback,
        error_callback=error_callback,
        retry_callback=retry_callback,
        write_options=write_options
    )

    try:
        client = InfluxDBClient3(
            token=INFLUXDB_TOKEN,
            host=INFLUXDB_HOST,
            database=INFLUXDB_BUCKET,
            org=INFLUXDB_ORG,
            write_client_options=wco
        )
        
        # Test connection
        test_query = f"SELECT COUNT(*) FROM {INFLUXDB_BUCKET} LIMIT 1"
        try:
            result = client.query(test_query)
            print("🔗 InfluxDB connection successful")
        except:
            print("ℹ️  InfluxDB connected (no existing data)")
        
        return client
        
    except Exception as e:
        print(f"❌ InfluxDB connection failed: {e}")
        print("🔧 Troubleshooting:")
        print("   1. Ensure Docker containers are running: docker-compose ps")
        print("   2. Check InfluxDB logs: docker-compose logs influxdb")
        print("   3. Verify network connectivity: docker-compose exec jupyter ping influxdb")
        return None

# Setup InfluxDB connection
influx_client = setup_influxdb_connection()
```

## Data Loading and Storage

```python
class DockerInfluxDBManager:
    """
    Docker-optimized InfluxDB manager
    """
    
    def __init__(self, client):
        self.client = client
        
    def write_characteristics_data(self, df: pd.DataFrame, 
                                 measurement_name: str = "jkp_characteristics"):
        """
        Write characteristics data to InfluxDB
        """
        if self.client is None:
            print("❌ No InfluxDB client available")
            return False
            
        try:
            # Define tag columns for efficient querying
            tag_columns = []
            for col in ["size_grp", "gvkey", "permno", "factor_model", "data_source"]:
                if col in df.columns:
                    tag_columns.append(col)
            
            print(f"📝 Writing {len(df)} records to {measurement_name}")
            print(f"🏷️  Tag columns: {tag_columns}")
            
            # Write to InfluxDB
            self.client.write(
                df,
                data_frame_measurement_name=measurement_name,
                data_frame_tag_columns=tag_columns
            )
            
            print(f"✅ Successfully wrote data to InfluxDB")
            return True
            
        except Exception as e:
            print(f"❌ Error writing to InfluxDB: {e}")
            return False
    
    def query_recent_data(self, limit: int = 100):
        """
        Query recent data to verify storage
        """
        if self.client is None:
            return None
            
        query = f"""
        SELECT *
        FROM jkp_characteristics
        ORDER BY time DESC
        LIMIT {limit}
        """
        
        try:
            result = self.client.query(query)
            print(f"📊 Retrieved {len(result)} recent records")
            return result
        except Exception as e:
            print(f"❌ Query failed: {e}")
            return None
    
    def get_data_summary(self):
        """
        Get summary statistics of stored data
        """
        if self.client is None:
            return None
            
        summary_query = """
        SELECT 
            COUNT(*) as total_records,
            COUNT(DISTINCT gvkey) as unique_companies,
            MIN(time) as earliest_date,
            MAX(time) as latest_date
        FROM jkp_characteristics
        """
        
        try:
            summary = self.client.query(summary_query)
            print("📈 Data Summary:")
            for _, row in summary.iterrows():
                print(f"   📊 Total records: {row['total_records']}")
                print(f"   🏢 Unique companies: {row['unique_companies']}")
                print(f"   📅 Date range: {row['earliest_date']} to {row['latest_date']}")
            return summary
        except Exception as e:
            print(f"❌ Summary query failed: {e}")
            return None

# Initialize InfluxDB manager
if influx_client:
    influx_manager = DockerInfluxDBManager(influx_client)
    print("🎯 InfluxDB Manager ready")
else:
    influx_manager = None
    print("⚠️  InfluxDB Manager not available")
```

## Load and Process Data

```python
# Load shared characteristics data
print("🔄 Loading shared characteristics data...")
characteristics_data = data_loader.load_shared_characteristics("characteristics.csv")

if not characteristics_data.empty:
    print("📊 Data loaded successfully!")
    print(f"Shape: {characteristics_data.shape}")
    print(f"Columns: {list(characteristics_data.columns)[:10]}...")  # First 10 columns
    
    # Prepare for InfluxDB
    print("\n🔧 Preparing data for InfluxDB...")
    processed_data = data_loader.prepare_for_influxdb(
        characteristics_data, 
        factor_model='shared_characteristics'
    )
    
    # Write to InfluxDB
    if influx_manager:
        print("\n📝 Writing to InfluxDB...")
        success = influx_manager.write_characteristics_data(processed_data)
        
        if success:
            print("\n📈 Getting data summary...")
            influx_manager.get_data_summary()
            
            print("\n🔍 Querying recent data...")
            recent_data = influx_manager.query_recent_data(limit=5)
            if recent_data is not None and not recent_data.empty:
                print(recent_data.head())
        else:
            print("❌ Failed to write data to InfluxDB")
    else:
        print("⚠️  Skipping InfluxDB write (no connection)")
        
else:
    print("❌ No data loaded. Please check that characteristics.csv exists in shared-data/")
    print("📁 You can place your own characteristics data file there")
```

## Sample Analysis

```python
def run_sample_analysis():
    """
    Run a sample factor analysis on the loaded data
    """
    if influx_manager is None:
        print("⚠️  Cannot run analysis without InfluxDB connection")
        return
    
    print("🧮 Running sample factor analysis...")
    
    # Query cross-sectional data
    analysis_query = """
    SELECT gvkey, be_me, ret_12_1, gp_at, market_equity
    FROM jkp_characteristics
    WHERE time >= now() - 90d
      AND be_me IS NOT NULL
      AND ret_12_1 IS NOT NULL
      AND gp_at IS NOT NULL
    ORDER BY time DESC
    LIMIT 1000
    """
    
    data = influx_manager.client.query(analysis_query)
    
    if data is not None and not data.empty:
        print(f"📊 Analysis dataset: {len(data)} observations")
        
        # Basic statistics
        print("\n📈 Factor Statistics:")
        stats = data[['be_me', 'ret_12_1', 'gp_at']].describe()
        print(stats)
        
        # Correlation matrix
        print("\n🔗 Factor Correlations:")
        corr = data[['be_me', 'ret_12_1', 'gp_at']].corr()
        print(corr)
        
        return data
    else:
        print("❌ No data available for analysis")
        return None

# Run sample analysis
sample_data = run_sample_analysis()
```

## Usage Instructions

### 1. Data Preparation
Place your factor characteristics data in `/shared-data/characteristics.csv` with columns:
- `eom`: End-of-month date
- `gvkey`: Company identifier
- `permno`: CRSP permanent number
- Factor characteristics (e.g., `be_me`, `ret_12_1`, `gp_at`)

### 2. Running the Notebook
1. Start Docker environment: `docker-compose up -d`
2. Access Jupyter: http://localhost:8888 (token: `qtab_jupyter_2024`)
3. Run this notebook to load data into InfluxDB

### 3. Querying Data
Use the InfluxDB web interface (http://localhost:8086) or Python client to query the stored data.

### 4. Custom Analysis
Modify the analysis functions to suit your specific research needs.

## Next Steps

1. **Add more data sources**: Place additional CSV files in `/shared-data/`
2. **Custom factor models**: Modify `FACTOR_MODELS` dictionary
3. **Advanced analytics**: Use the stored data for factor model estimation
4. **Real-time updates**: Set up automated data refresh workflows

## Troubleshooting

### Common Issues
- **Container connectivity**: Ensure all containers are running with `docker-compose ps`
- **Data not found**: Check that files exist in `/shared-data/` directory
- **InfluxDB errors**: Review container logs with `docker-compose logs influxdb`

### Getting Help
- Review the [Docker Setup Guide](Docker_Setup_README.md)
- Check the [InfluxDB Query Guide](InfluxDB_Query_Guide.md)
- Examine container logs for detailed error messages